# Module C, Week 5: Support Vector Machines and the Kernel Trick
### Integrated Capstone Project: Predicting Corporate Bankruptcy from Financial Ratios
**Astha Goel**

---

## Project context

This capstone predicts corporate bankruptcy from published financial ratios, running the same pipeline
separately on two datasets and comparing results rather than merging them:

| | Poland | Taiwan |
|---|---|---|
| Observations | 43,405 firm-years (2000 to 2013) | 6,819 firms (1999 to 2009) |
| Features | 64 financial ratios | 95 financial ratios |
| Bankrupt | 2,091 (4.8%) | 220 (3.2%) |

**Established so far.** Module B: no single ratio predicts bankruptcy alone, PCA does not compress these data
(PC1 captures only 15.3% and 13.4% of variance), yet a random forest reaches 0.9395 (Poland) and 0.9482 (Taiwan).
Week 1: severe multicollinearity (a Taiwanese pair at VIF ≈ 10¹⁵), and added flexibility helps Poland and does not help Taiwan. Week 2: features ranging to 10¹⁰ break linear models until winsorized; lasso removes redundant
ratio families automatically. Week 4: scaler choice swings Poland's ROC-AUC by 0.21, `QuantileTransformer` wins
both datasets, and `class_weight='balanced'` degrades calibration without improving ranking.

## This week

Week 5 covers support vector machines, the kernel trick, and SVM regularization. Beyond applying the method,
this notebook uses it to run a **test of a specific hypothesis** formed in earlier weeks.

Weeks 1 and 2 both found that added flexibility helps Poland and does not help Taiwan. Polynomial terms raised Poland's test R² while lowering Taiwan's, and interaction terms gave Poland a large, repeatable gain while leaving Taiwan unchanged. The explanation
offered was that Polish bankruptcy risk depends on interactions between ratios while Taiwanese risk is close to
additive.

SVM kernels provide an independent way to test that. The kernel controls exactly how flexible the decision boundary can be, and it does so by a completely different route than polynomial feature expansion. If the hypothesis is
right, Taiwan should prefer the linear kernel and Poland should prefer a flexible one.

## 1. Setup and loading

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import QuantileTransformer
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 110
RANDOM_STATE = 42

# rank-based scaling, selected in Week 4 after comparing six scalers
qt = lambda: QuantileTransformer(output_distribution='normal', random_state=RANDOM_STATE)

from sklearn.svm import SVC
from sklearn.metrics import confusion_matrix, classification_report

In [ ]:
# ── TAIWAN ────────────────────────────────────────────────────────────────────
df_taiwan = pd.read_csv('Taiwan_data_capstone.csv')
df_taiwan.columns = df_taiwan.columns.str.strip()   # headers carry a leading space
df_taiwan['Bankrupt?'] = pd.to_numeric(df_taiwan['Bankrupt?'], errors='coerce').astype(int)

numeric_cols_taiwan = [c for c in df_taiwan.select_dtypes(include='number').columns
                       if c != 'Bankrupt?']
for col in numeric_cols_taiwan:
    df_taiwan[col] = df_taiwan[col].fillna(df_taiwan[col].median())
numeric_cols_taiwan = [c for c in numeric_cols_taiwan if df_taiwan[c].nunique() > 1]

print(f"Taiwan: {df_taiwan.shape}, {len(numeric_cols_taiwan)} usable features")
print(df_taiwan['Bankrupt?'].value_counts().to_dict())

In [ ]:
# ── POLAND ────────────────────────────────────────────────────────────────────
df_polish = pd.read_csv('Polish_dataset_capstone.csv')

column_names_64 = [
    'net_profit_to_total_assets', 'total_liabilities_to_total_assets',
    'working_capital_to_total_assets', 'current_assets_to_short_term_liabilities',
    'cash_to_operating_expenses', 'retained_earnings_to_total_assets',
    'EBIT_to_total_assets', 'book_value_of_equity_to_total_liabilities',
    'sales_to_total_assets', 'equity_to_total_assets',
    'gross_profit_to_short_term_liabilities', 'gross_profit_to_total_assets',
    'gross_profit_to_sales', 'gross_profit_to_total_assets_2',
    'working_capital_to_fixed_assets', 'receivables_to_total_assets',
    'liabilities_to_total_assets_duration', 'EBIT_to_total_assets_2',
    'sales_to_total_assets_2', 'working_capital_to_sales',
    'sales_to_inventories', 'net_profit_to_sales',
    'gross_profit_to_total_assets_3', 'current_liabilities_to_equity',
    'short_term_liabilities_to_equity', 'working_capital',
    'depreciation_to_total_assets', 'inventories_to_sales',
    'net_profit_to_inventories', 'total_liabilities_to_equity',
    'profit_on_sales_to_total_assets', 'total_liabilities_to_cash',
    'operating_expenses_to_short_term_liabilities', 'operating_expenses_to_total_liabilities',
    'profit_to_total_assets', 'net_profit_to_total_assets_2',
    'total_liabilities_to_operating_cash', 'total_liabilities_to_total_assets_rotation',
    'profit_to_long_term_liabilities', 'total_assets_to_total_liabilities',
    'short_term_liabilities_to_long_term_liabilities', 'short_term_liabilities_to_equity_2',
    'long_term_liabilities_to_equity', 'sales_to_short_term_liabilities',
    'inventory_to_receivables', 'receivables_to_sales',
    'net_profit_to_total_assets_3', 'total_costs_to_total_sales',
    'long_term_liabilities_to_total_assets', 'sales_to_equity',
    'sales_to_total_assets_3', 'liabilities_to_working_capital',
    'operating_cash_to_liabilities', 'operating_cash_to_total_liabilities',
    'operating_cash_to_sales', 'sales_to_receivables',
    'operating_expenses_to_sales', 'total_costs_to_total_assets',
    'profit_on_operating_activities', 'depreciation_to_profit',
    'total_liabilities_to_profit', 'profit_on_operating_activities_to_financial_expenses',
    'working_capital_to_liabilities', 'equity_to_fixed_assets'
]
df_polish = df_polish.rename(columns={f'A{i+1}': column_names_64[i] for i in range(64)})

# force ratio columns numeric, else select_dtypes silently skips any that arrive as strings
for c in column_names_64:
    if c in df_polish.columns:
        df_polish[c] = pd.to_numeric(df_polish[c], errors='coerce')
df_polish['class'] = pd.to_numeric(df_polish['class'], errors='coerce').astype(int)

numeric_cols_polish = [c for c in df_polish.select_dtypes(include='number').columns
                       if c not in ['class', 'year', 'id']]
for col in numeric_cols_polish:
    df_polish[col] = df_polish[col].fillna(df_polish[col].median())
numeric_cols_polish = [c for c in numeric_cols_polish if df_polish[c].nunique() > 1]

print(f"Poland: {df_polish.shape}, {len(numeric_cols_polish)} usable features")
print(df_polish['class'].value_counts().to_dict())

## 2. What an SVM does, and why scaling is mandatory

An SVM finds the decision boundary with the widest possible margin between classes, defined by the handful of
points sitting closest to it, the support vectors. Points far from the boundary have no influence at all, which
makes SVMs quite different from logistic regression, where every observation contributes to the fit.

Scaling is not optional. The margin is a geometric distance, so a feature measured in billions would dominate
that distance calculation entirely and determine the boundary by itself. Given that these datasets contain
features reaching 10¹⁰ (Week 2), an unscaled SVM here would be meaningless. The `QuantileTransformer` selected in
Week 4 is used throughout.

Poland is subsampled to 8,000 rows: kernel SVMs scale roughly quadratically to cubically with sample size, so
43,405 rows is impractical. This is a stated limitation on the Polish figures, not a modelling choice.

In [ ]:
def fit_svm(df, feats, target, name, kernel='rbf', C=1.0, n_sample=None, **kw):
    d = df.sample(n_sample, random_state=RANDOM_STATE) if n_sample and n_sample < len(df) else df
    X, y = d[feats], d[target].values
    Xtr, Xte, ytr, yte = train_test_split(
        X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y)
    pipe = make_pipeline(qt(), SVC(kernel=kernel, C=C, probability=True,
                                   random_state=RANDOM_STATE, **kw)).fit(Xtr, ytr)
    auc = roc_auc_score(yte, pipe.predict_proba(Xte)[:, 1])
    n_sv = pipe[-1].n_support_.sum()
    return auc, n_sv, len(Xtr), pipe, (Xte, yte)

N_POLAND = 8000   # kernel SVMs scale poorly; stated limitation on Polish results

## 3. The kernel trick

The kernel trick lets an SVM fit a nonlinear boundary without ever constructing the nonlinear features
explicitly. Instead of transforming the data into a higher-dimensional space and working there, which would be
expensive or infinite-dimensional, a kernel function computes inner products *as if* the data had been
transformed. The cost stays proportional to the number of observations rather than the number of implied
features.

Three kernels are compared:

- **Linear**: a straight hyperplane, equivalent to no transformation
- **RBF (Gaussian)**: a smooth, locally flexible boundary, implicitly infinite-dimensional
- **Polynomial (degree 3)**: explicit cubic interactions, the kernel analogue of what Week 1 did by hand

In [ ]:
def kernel_comparison(df, feats, target, name, n_sample=None):
    rows = []
    for label, kw in [('Linear', dict(kernel='linear')),
                      ('RBF', dict(kernel='rbf', gamma='scale')),
                      ('Polynomial (d=3)', dict(kernel='poly', degree=3))]:
        auc, n_sv, n_tr, _, _ = fit_svm(df, feats, target, name, C=1.0,
                                        n_sample=n_sample, **kw)
        rows.append({'Kernel': label, 'Test ROC-AUC': auc,
                     'Support vectors': n_sv, '% of training set': 100 * n_sv / n_tr})
    res = pd.DataFrame(rows)
    print(f"\n=== {name}: kernel comparison (C = 1.0) ===")
    print(res.to_string(index=False, float_format=lambda x: f"{x:.4f}"))
    return res

ker_tw = kernel_comparison(df_taiwan, numeric_cols_taiwan, 'Bankrupt?', 'TAIWAN')
ker_pl = kernel_comparison(df_polish, numeric_cols_polish, 'class', 'POLAND', n_sample=N_POLAND)

### Taiwan prefers the simplest kernel

In Taiwan the **linear** kernel wins outright, beating RBF and the cubic polynomial. That is a meaningful result
rather than a shrug: given the freedom to curve, the SVM does better refusing it. In Poland the linear kernel
also leads at this setting, but the regularization sweep below shows that changes once the RBF is allowed enough
flexibility.

The support vector counts are worth noting too. A model using a large fraction of the training set as support
vectors is one that finds the classes hard to separate. The boundary is being pinned down by many points rather than a clean few.

## 4. Regularization: the C parameter

C controls the trade-off between margin width and training errors, and it is the SVM's regularization dial:

- **Small C**: strong regularization. Prioritizes a wide margin, tolerates misclassified training points, gives
  a smoother boundary that may underfit.
- **Large C**: weak regularization. Penalizes every training error heavily, producing an intricate boundary that
  may overfit.

This is the parameter that directly tests the hypothesis. If Poland's signal lives in interactions, Poland should
benefit from a larger C on the RBF kernel, which permits a more intricate boundary. If Taiwan's is additive, the
extra flexibility should not help.

In [ ]:
def C_sweep(df, feats, target, name, Cs=(0.1, 1.0, 10.0), n_sample=None):
    rows = []
    for C in Cs:
        auc, n_sv, n_tr, _, _ = fit_svm(df, feats, target, name, kernel='rbf',
                                        C=C, gamma='scale', n_sample=n_sample)
        rows.append({'C': C, 'Test ROC-AUC': auc, 'Support vectors': n_sv,
                     '% of training set': 100 * n_sv / n_tr})
    res = pd.DataFrame(rows)
    print(f"\n=== {name}: RBF regularization sweep ===")
    print(res.to_string(index=False, float_format=lambda x: f"{x:.4f}"))
    return res

C_tw = C_sweep(df_taiwan, numeric_cols_taiwan, 'Bankrupt?', 'TAIWAN')
C_pl = C_sweep(df_polish, numeric_cols_polish, 'class', 'POLAND', n_sample=N_POLAND)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

ax = axes[0]
x = np.arange(3); w = 0.37
ax.bar(x - w/2, ker_pl['Test ROC-AUC'], w, label='Poland', color='steelblue')
ax.bar(x + w/2, ker_tw['Test ROC-AUC'], w, label='Taiwan', color='coral')
ax.set_xticks(x); ax.set_xticklabels(ker_tw['Kernel'], fontsize=9)
ax.set_ylim(0.70, 1.0); ax.set_ylabel('Test ROC-AUC')
ax.set_title('Kernel comparison at C = 1', fontsize=10); ax.legend(fontsize=9)

ax = axes[1]
ax.plot(C_pl['C'], C_pl['Test ROC-AUC'], marker='o', ms=7, color='steelblue', label='Poland')
ax.plot(C_tw['C'], C_tw['Test ROC-AUC'], marker='s', ms=7, color='coral', label='Taiwan')
ax.set_xscale('log'); ax.set_xlabel('C (weaker regularization ->)')
ax.set_ylabel('Test ROC-AUC')
ax.set_title('RBF: more flexibility helps Poland, not Taiwan', fontsize=10)
ax.legend(fontsize=9)
plt.tight_layout(); plt.show()

### The hypothesis holds

Raising C on the RBF kernel lifts Poland substantially: enough that the flexible RBF overtakes Poland's linear
kernel. Taiwan gains far less, and never reaches what its plain linear kernel already achieved.

This replicates the Week 1 and Week 2 finding using a completely different method. Polynomial expansion and
kernel selection have nothing in common in how they work, yet both say the same thing: **Poland needs a nonlinear decision
boundary and Taiwan does not.** Two independent methods agreeing is much stronger evidence than either
alone, because it rules out the possibility that the result was an artifact of how one of them works.

## 5. Where the SVM actually errs

ROC-AUC measures ranking across all thresholds. The confusion matrix shows what happens at the one threshold
actually used, which is what a deployed model would do.

In [ ]:
for name, df, feats, target, n in [('TAIWAN', df_taiwan, numeric_cols_taiwan, 'Bankrupt?', None),
                                   ('POLAND', df_polish, numeric_cols_polish, 'class', N_POLAND)]:
    best_C = float(C_tw.loc[C_tw['Test ROC-AUC'].idxmax(), 'C']) if name == 'TAIWAN' \
             else float(C_pl.loc[C_pl['Test ROC-AUC'].idxmax(), 'C'])
    auc, n_sv, n_tr, pipe, (Xte, yte) = fit_svm(df, feats, target, name, kernel='rbf',
                                                C=best_C, gamma='scale', n_sample=n)
    pred = pipe.predict(Xte)
    tn, fp, fn, tp = confusion_matrix(yte, pred, labels=[0, 1]).ravel()
    print(f"\n=== {name}: RBF SVM at C={best_C}, ROC-AUC={auc:.4f} ===")
    print(f"   True negatives {tn:5d}   False positives {fp:4d}")
    print(f"   False negatives {fn:4d}   True positives  {tp:4d}")
    rec = tp / (tp + fn) if (tp + fn) else 0
    print(f"   Recall on bankrupt firms: {rec:.3f}  ({tp} of {tp+fn} caught)")

The recall figures expose the same problem Module B found with random forests: at the default threshold the
model catches only a minority of bankruptcies, because the decision boundary is positioned to minimize overall
errors and bankruptcies are rare. Week 4 showed that moving the threshold against an explicit cost function fixes
this, and the same remedy applies here.

## 6. Conclusions

1. **The kernel trick confirms the project's central hypothesis through independent machinery.** Taiwan's best
   kernel is linear; Poland's is a flexible RBF once C permits it. This agrees with the polynomial and
   interaction results from Weeks 1 and 2, which have nothing in common in how they work with kernel methods.

2. **Taiwan actively prefers the simplest kernel.** Given the option to curve, the SVM performs worse for taking
   it, the cubic polynomial is the weakest of the three. This is positive evidence of additivity, not merely an
   absence of evidence for nonlinearity.

3. **C behaves the way the theory says, with opposite results in the two countries.** Weakening regularization
   produces a more intricate boundary, which Poland rewards and Taiwan does not.

4. **SVMs are not the right production choice here.** They matched but did not beat the Week 4 logistic
   regression in Taiwan, required subsampling in Poland, and at the default threshold their recall on bankrupt
   firms is poor. Their value in this project is diagnostic: as a second, independent check of the functional
   form.

**Carried forward:** the functional-form difference is now supported by two independent method families. For the
cross-country transfer experiment this predicts poor transfer regardless of how carefully the ratio families are
mapped, since the two problems differ in shape and not merely in coefficients.